In [1]:
import os
import math
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
documents = [
    "Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.",
    "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.",
    "GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.",
    "RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.",
    "FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워크입니다.",
    "트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.",
    "FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러리입니다.",
    "프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.",
    "임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.",
]


In [2]:
# 파이썬은 뭔가요? -> Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.
#                     파이썬 ~~~

In [3]:
doc_embeddings = embeddings.embed_documents(documents)
len(doc_embeddings), len(doc_embeddings[0])

(10, 1536)

In [4]:
# doc_embeddings.shape

In [5]:
import numpy as np
np.array(doc_embeddings).shape

(10, 1536)

In [6]:
np.array(doc_embeddings)

array([[ 0.0065918 ,  0.0304718 , -0.02268982, ..., -0.05459595,
        -0.01646423,  0.00078058],
       [ 0.0022049 ,  0.001441  , -0.01389313, ..., -0.02348328,
         0.01748657, -0.01128387],
       [-0.02810669, -0.00686646, -0.01013947, ..., -0.01411438,
         0.04055786, -0.01038361],
       ...,
       [-0.02555847, -0.0003047 , -0.01858521, ...,  0.03930664,
        -0.04626465, -0.01712036],
       [ 0.03399658,  0.04638672, -0.02185059, ...,  0.00662994,
         0.02731323, -0.01065826],
       [-0.01887512,  0.0411377 ,  0.00096655, ..., -0.01397705,
         0.00841522,  0.00332832]], shape=(10, 1536))

In [7]:
# !pip install sentence-transformers

In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding = embedding_model.encode(documents)
embedding.shape

(10, 384)

In [10]:
embedding

array([[-0.02818309,  0.02315177, -0.01332168, ...,  0.09440271,
         0.06264333, -0.00686623],
       [ 0.00433091,  0.02231268,  0.06852957, ...,  0.06287049,
         0.03236052,  0.01451786],
       [ 0.04224599,  0.06530686,  0.02390076, ...,  0.01747432,
        -0.08692621,  0.02379501],
       ...,
       [-0.03313585,  0.02055405,  0.00575816, ...,  0.0755645 ,
        -0.08282121, -0.0617734 ],
       [ 0.09051032, -0.00739942,  0.04021923, ..., -0.00768866,
        -0.11420648, -0.01269378],
       [ 0.01094468,  0.08185508,  0.03489941, ...,  0.01663297,
        -0.08264522,  0.02735936]], shape=(10, 384), dtype=float32)

In [11]:
# 파이썬 뭔가요?  -> Python 은 ~~~
# ABCDEF 뭔가요 ABCDEF ?  -> abcde 뭔가요?

In [12]:
def keyword_search(query, docs, top_k=3):
    query_tokens = set(query.lower().split())
    scores = []
    for i, doc in enumerate(docs):
        doc_tokens = set(doc.lower().split())
        overlap = len(query_tokens & doc_tokens)
        scores.append((i, overlap))
    
    # scores = [(0, 3), (1, 2), (2, 5) ...]
    scores.sort(key=lambda x:x[1], reverse=True)    # [(2, 5), (0, 3), (1, 2) ... ]
    return scores[:top_k]

In [13]:
results = keyword_search("Python 프로그래밍 언어", documents)
print(results)
for idx, scores in results:
    print(f"[{idx}] overlap = {scores} | {documents[idx][:30]} ") 

[(0, 1), (3, 1), (1, 0)]
[0] overlap = 1 | Python은 데이터 과학과 머신러닝에 널리 사용되는  
[3] overlap = 1 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로  
[1] overlap = 0 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 


In [14]:
def vector_search(query, docs, doc_embs, top_k=3):
    q_emb = np.array(embeddings.embed_query(query))
    similarities = np.dot(doc_embs, q_emb) / (np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(q_emb))
    top_indices = similarities.argsort()[::-1][:top_k]
    return [(i, similarities[i]) for i in top_indices]

results = vector_search("Python 프로그래밍 언어", documents, np.array(doc_embeddings), top_k=3)

In [15]:
results
for idx, scores in results:
    print(f"[{idx}] similarity = {scores} | {documents[idx][:30]} ")

[0] similarity = 0.5529189499619658 | Python은 데이터 과학과 머신러닝에 널리 사용되는  
[1] similarity = 0.3717890483207657 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 
[3] similarity = 0.3603723550951089 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로  


In [16]:
results = vector_search('FAISS', documents, np.array(doc_embeddings), top_k=3)

In [17]:
for idx, scores in results:
    print(f"[{idx}] similarity = {scores} | {documents[idx][:30]} ")

[7] similarity = 0.527924286216923 | FAISS는 Facebook AI가 개발한 효율적인 유 
[4] similarity = 0.19232367271362005 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 
[5] similarity = 0.0962571308471338 | FastAPI는 Python으로 빠른 웹 API를 구축 


In [18]:
def overlap_rate(keyword_results, vector_results):
    
    kw_ids = set(idx for idx, _ in keyword_results)
    vec_ids = set(idx for idx, _ in vector_results)
    
    overlap = kw_ids & vec_ids  # set(0,1,3,4,5,7)
    union = kw_ids | vec_ids    # set()
    return len(overlap) / len(union)
    

In [19]:
for query in ['Python 프로그래밍', '딥러닝 모델', 'FAISS 라이브러리']:
    kw = keyword_search(query, documents, top_k=5)
    vec = vector_search(query, documents, np.array(doc_embeddings), top_k=5)
    rate = overlap_rate(kw, vec)
    print(f"{query} overlap : {rate}")

Python 프로그래밍 overlap : 0.6666666666666666
딥러닝 모델 overlap : 0.42857142857142855
FAISS 라이브러리 overlap : 0.25


In [20]:
kw

[(0, 0), (1, 0), (2, 0), (3, 0), (4, 0)]

In [21]:
vec

[(np.int64(7), np.float64(0.6645721103085903)),
 (np.int64(4), np.float64(0.24952828001098243)),
 (np.int64(2), np.float64(0.21770991370746465)),
 (np.int64(8), np.float64(0.20219650146536342)),
 (np.int64(6), np.float64(0.1955491901731619))]

In [22]:
def simple_hybrid(query, docs, doc_embs, top_k=3):
    kw = keyword_search(query, docs, top_k = len(docs))
    vec = vector_search(query, docs, np.array(doc_embs), top_k= len(docs))
    
    kw_scores = {idx : score for idx, score in kw}
    vec_scores = {idx : score for idx, score in vec}
    
    kw_max = max(kw_scores.values()) or 1
    vec_max = max(vec_scores.values()) or 1
    
    combined = {}
    
    for idx in range(len(docs)):
        kw_score = kw_scores.get(idx, 0) / kw_max
        vec_score = vec_scores.get(idx, 0) / vec_max
        
        combined[idx] = kw_score + vec_score
        
    ranked = sorted(combined.items(), key=lambda x:x[1], reverse=True)
    return ranked[:top_k]
    
#     kw : [(0, 100), (1, 50), (2, 30), ..... (100, 0)] -> [(0, 1), (1, 0.5), (2, 0.3) ...]
#     vec : [(0, 1),  (1, 0.5), (2, 0.9)...]            -> [(0, 1), (1, 0.5), (2, 0.9) ...]

In [23]:
for query in ['Python 프로그래밍 언어', '딥러닝 모델 구조', 'FAISS']:
    results = simple_hybrid(query, documents, np.array(doc_embeddings), top_k=3)
    
#     print(results)
    for idx, score in results:
        print(f"[{idx}] {score} | {documents[idx][:30]}")
        
    print("----------------------------------------------------")

[0] 2.0 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
[3] 1.6519023830556376 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 
[1] 0.6725032604143426 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고
----------------------------------------------------
[6] 2.0 | 트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 
[3] 0.49053808203235794 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 
[0] 0.4186472659347687 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
----------------------------------------------------
[7] 1.0 | FAISS는 Facebook AI가 개발한 효율적인 유
[4] 0.36430162001411437 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
[5] 0.18233131788065143 | FastAPI는 Python으로 빠른 웹 API를 구축
----------------------------------------------------


###  TF : f(t, d) / | d |    (f(t, d) : document에 나온 t의 개수, |d| : d 의 길이)
###  IDF : log(N / (t가 등장한 문서의 개수)) (N : 전체 문서수)

In [24]:
import math
class TFIDF:
    def __init__(self, documents):
        self.docs = documents
        self.tokenized = [doc.lower().split() for doc in documents]
        self.N = len(documents)
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):
                self.df[t] = self.df.get(t,0) + 1
                
    def tf(self, term, doc_tokens):
        return doc_tokens.count(term) / len(doc_tokens)
    
    def idf(self, term):
        return math.log(self.N / self.df.get(term, 1))
    
    def score(self, query, doc_idx):
        tokens = self.tokenized[doc_idx]
        return sum(self.tf(t, tokens) * self.idf(t) for t in query.lower().split())
    
    def search(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

In [25]:
tfidf = TFIDF(documents)
for idx, score in tfidf.search('Python 프로그래밍'):
    print(f"[{idx}] {score} | {documents[idx][:30]} ")

[0] 0.28782313662425574 | Python은 데이터 과학과 머신러닝에 널리 사용되는  
[1] 0.0 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 
[2] 0.0 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고  


### k1, b

In [26]:
class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = documents
        self.tokenized = [doc.lower().split() for doc in documents]
        self.N = len(documents)
        self.avgdl = sum(len(d) for d in self.tokenized) / self.N
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):
                self.df[t] = self.df.get(t, 0) + 1
                
    def idf(self, term):
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)
    
    def score(self, query, doc_idx):
        tokens = self.tokenized[doc_idx]
        dl = len(tokens)
        tf_counter = Counter(tokens)
        total = 0.0
        for t in query.lower().split():
            tf = tf_counter.get(t, 0)
            numerator = tf * (self.k1 + 1)
            denominator = tf + (self.k1  * (1-self.b + self.b * dl / self.avgdl))
            total += self.idf(t) * numerator / denominator
            
        return total
            
    def search(self, query, top_k = 3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x:x[1], reverse=True)[:top_k]
    

In [27]:
bm25 = BM25(documents)
for idx, score in bm25.search('Python 프로그래밍'):
    print(f"[{idx}] {score} | {documents[idx][:30]} ")

[0] 2.0360600223111596 | Python은 데이터 과학과 머신러닝에 널리 사용되는  
[1] 0.0 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 
[2] 0.0 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고  


In [28]:
from langchain_community.retrievers import BM25Retriever

In [29]:
# from langchain.retrievers import BM25Retriever
# from langchain_classics import BM25Retriever

In [30]:
from langchain_core.documents import Document

In [31]:
docs_lc = [Document(page_content = d, metadata = {"index":i}) for i, d in enumerate(documents)]
bm25_retriever = BM25Retriever.from_documents(docs_lc)

In [32]:
bm25_retriever.k = 3

In [33]:
results = bm25_retriever.invoke('Python 프로그래밍')
results

[Document(metadata={'index': 0}, page_content='Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.'),
 Document(metadata={'index': 9}, page_content='임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.'),
 Document(metadata={'index': 8}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.')]

In [34]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(docs_lc, embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs = {'k' : 3})

In [35]:
results = vector_retriever.invoke('딥러닝 모델 구조')
results

[Document(id='05d9d7a6-103e-430a-b8df-c355fc3ce8c4', metadata={'index': 6}, page_content='트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.'),
 Document(id='bc750420-ee3d-42a9-af00-8513287e977e', metadata={'index': 3}, page_content='GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.'),
 Document(id='4cd6effb-b568-4b7f-8c92-4337f3a41fc9', metadata={'index': 4}, page_content='RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.')]

In [36]:
from langchain_classic.retrievers import EnsembleRetriever

In [37]:
ensemble = EnsembleRetriever(
                retrievers = [vector_retriever, bm25_retriever],
                weights = [0.5, 0.5],
            )

In [38]:
results = ensemble.invoke("Python 데이터 과학")
results

[Document(id='ff62a75b-9c95-4844-ba95-7f273637a1f8', metadata={'index': 0}, page_content='Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.'),
 Document(id='4a4aa421-4104-4da1-9dfa-397cf4e45970', metadata={'index': 9}, page_content='임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.'),
 Document(id='3aea6cb4-b060-4a76-ad45-04c87490cade', metadata={'index': 2}, page_content='벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.'),
 Document(metadata={'index': 8}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.')]

In [39]:
for w_vec, w_bm25 in [(0.2, 0.8), (0.5, 0.5), (0.8, 0.2)]:
    ensemble = EnsembleRetriever(
                retrievers = [vector_retriever, bm25_retriever],
                weights = [w_vec, w_bm25],
            )
    results = ensemble.invoke("Python 데이터 과학")
    top_idx = results[0].metadata['index']
    print(f"BM25 = {w_bm25}, Vector = {w_vec} -> Top-1 : {top_idx}, {results[0].page_content[:30]}")

BM25 = 0.8, Vector = 0.2 -> Top-1 : 0, Python은 데이터 과학과 머신러닝에 널리 사용되는 
BM25 = 0.5, Vector = 0.5 -> Top-1 : 0, Python은 데이터 과학과 머신러닝에 널리 사용되는 
BM25 = 0.2, Vector = 0.8 -> Top-1 : 0, Python은 데이터 과학과 머신러닝에 널리 사용되는 


In [40]:
queries = ["Python 프로그래밍 언어", "딥러닝 모델 구조", "FAISS 라이브러리"]
for q in queries:
    bm25_res = bm25_retriever.invoke(q)
    vec_res = vector_retriever.invoke(q)
    ens_res = ensemble.invoke(q)
    
    bm25_ids = [d.metadata['index'] for d in bm25_res]
    vec_ids = [d.metadata['index'] for d in vec_res]
    ens_ids = [d.metadata['index'] for d in ens_res]
    
    print(q)
    print(f"BM25 : {bm25_ids}, vec : {vec_ids}, ens : {ens_ids}")

Python 프로그래밍 언어
BM25 : [0, 3, 8], vec : [0, 1, 3], ens : [0, 3, 1, 8]
딥러닝 모델 구조
BM25 : [6, 9, 8], vec : [6, 3, 4], ens : [6, 3, 4, 9, 8]
FAISS 라이브러리
BM25 : [9, 8, 7], vec : [7, 4, 2], ens : [7, 4, 2, 9, 8]


In [ ]:
# 점수 -> 순위
# reciprocal (1/n) rank fusion
# 1등 : 1 : 100
# 2등 : 2 : 90
        
# 1등 : 5 : 0.9
# 2등 : 8 : 0.88

        
#       1,2        

In [ ]:
# RRF (Reciprocal Rank Fusion) : sum( 1 / (k+ rank) ) 
# # temperature  -> 커지면 : 확률분포가 평평해져서 -> 엉뚱한 토큰도 잘나오게  / 작아지면 : 
# k : 1 : 1등 = 1/2 : 0.5   2등 = 1/3 : 0.3333
# k : 100 : 1등 = 1/101     2등 = 1/102  

# 일부 검색결과의 이상치에 robust 
# Elasticsearch

In [43]:
def rrf(rankings, k=60):
    rrf_scores = {}
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id,0) + 1 / (k + rank)
            
    return sorted(rrf_scores.items(), key=lambda x : x[1], reverse=True)

In [44]:
query = 'Python 데이터 과학'
bm25_results = bm25.search(query, top_k=5)
tfidf_results = tfidf.search(query, top_k=5)

In [45]:
combined = rrf([bm25_results, tfidf_results], k=60)

In [46]:
combined

[(0, 0.03278688524590164),
 (1, 0.03225806451612903),
 (2, 0.031746031746031744),
 (3, 0.03125),
 (4, 0.03076923076923077)]

In [47]:
for doc_id, score in combined[:3]:
    print(f" [{doc_id}] : {score} | {documents[doc_id][:30]}")

 [0] : 0.03278688524590164 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
 [1] : 0.03225806451612903 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고
 [2] : 0.031746031746031744 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 


In [49]:
query = '대규모 언어 모델'
# bm25
# vector retriever

# rrf 결과 출력
bm25_r = bm25.search(query, top_k=3)
vector_r = vector_search(query, documents, np.array(doc_embeddings), top_k= 3)

In [51]:
for rank, (idx, score) in enumerate(bm25_r, 1):
    print(f"{rank}위 : [{idx}] {documents[idx][:30]}, (score = {score})")

print('================================')

for rank, (idx, score) in enumerate(vector_r, 1):
    print(f"{rank}위 : [{idx}] {documents[idx][:30]}, (score = {score})")

1위 : [3] GPT-4는 OpenAI가 개발한 대규모 언어 모델로 , (score = 3.8607643329637216)
2위 : [6] 트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 , (score = 2.0360600223111596)
3위 : [0] Python은 데이터 과학과 머신러닝에 널리 사용되는 , (score = 0.0)
1위 : [3] GPT-4는 OpenAI가 개발한 대규모 언어 모델로 , (score = 0.5316671092497184)
2위 : [1] 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고, (score = 0.30308628627269846)
3위 : [6] 트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 , (score = 0.2971697967202345)


In [52]:
k = 60
combined = rrf([bm25_r, vector_r], k= k)
for doc_id, score in combined[:3]:
    print(f" [{doc_id}] : {score} | {documents[doc_id][:30]}")

 [3] : 0.03278688524590164 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 
 [6] : 0.03200204813108039 | 트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 
 [1] : 0.016129032258064516 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고


In [ ]:
# 그리드 서치


In [53]:
eval_dataset = [
    {"query": "Python 프로그래밍 언어", "relevant": [0, 5]},
    {"query": "자연어 처리 NLP 기술", "relevant": [1]},
    {"query": "벡터 데이터베이스", "relevant": [2, 7]},
    {"query": "GPT 대규모 언어 모델", "relevant": [3]},
    {"query": "RAG 검색 증강 생성", "relevant": [4]},
    {"query": "트랜스포머 어텐션", "relevant": [6]},
    {"query": "임베딩 벡터 변환", "relevant": [9]},
]

In [54]:
def precision_at_k(retrieved, relevant, k):
    """상위 K개 중 정답 비율"""
    top_k = retrieved[:k]
    return len(set(top_k) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    """정답 중 상위 K개에 포함된 비율"""
    top_k = retrieved[:k]
    return len(set(top_k) & set(relevant)) / len(relevant) if relevant else 0

def mrr(retrieved, relevant):
    """첫 정답의 순위 역수"""
    for rank, doc_id in enumerate(retrieved, 1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

In [55]:
retrieved = [3, 1,4,0,2]
relevant = [4, 0]
print(precision_at_k(retrieved, relevant, k=3), recall_at_k(retrieved, relevant, 3), mrr(retrieved, relevant))

0.3333333333333333 0.5 0.3333333333333333


In [57]:
def hybrid_search(query, w_bm25 = 0.5, top_k=5):
    bm25_r = bm25.search(query, top_k=top_k)
    vec_r = vector_search(query, documents, np.array(doc_embeddings), top_k= top_k)
    
    rrf_scores = {}
    for rank, (doc_id, _) in enumerate(bm25_r, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id,0) + w_bm25 / (60 + rank)
    for rank, (doc_id, _) in enumerate(vec_r, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id,0) + (1 - w_bm25) / (60 + rank) 
            
    ranked = sorted(rrf_scores.items(), key=lambda x : x[1], reverse=True)
    return [doc_id for doc_id, _ in ranked[:top_k]]

In [58]:
weights = np.arange(0, 1.05, 0.1)
weights

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

In [60]:
# grid search
results = []
for w in weights:
    mrr_sum = 0
    for item in eval_dataset:
        retrieved = hybrid_search(item['query'], w_bm25=w, top_k=5)
        mrr_sum += mrr(retrieved, item['relevant'])
    avg_mrr = mrr_sum / len(eval_dataset)
    results.append((w, avg_mrr))
    print(f" w_bm25 = {w}: MRR ={avg_mrr}")
          
best_w, best_mrr = max(results, key = lambda x: x[1])
print(f"optimized weight : {best_w} (MRR = {best_mrr})")

 w_bm25 = 0.0: MRR =1.0
 w_bm25 = 0.1: MRR =0.9047619047619048
 w_bm25 = 0.2: MRR =0.9047619047619048
 w_bm25 = 0.30000000000000004: MRR =0.9047619047619048
 w_bm25 = 0.4: MRR =0.9047619047619048
 w_bm25 = 0.5: MRR =0.8333333333333333
 w_bm25 = 0.6000000000000001: MRR =0.7857142857142857
 w_bm25 = 0.7000000000000001: MRR =0.7857142857142857
 w_bm25 = 0.8: MRR =0.7857142857142857
 w_bm25 = 0.9: MRR =0.7857142857142857
 w_bm25 = 1.0: MRR =0.7857142857142857
optimized weight : 0.0 (MRR = 1.0)


In [ ]:
# '파이썬 뭔가요?'

# '파이썬은 컴퓨터 언어입니다'

In [63]:
# !pip install kiwipiepy
from kiwipiepy import Kiwi

In [64]:
kiwi = Kiwi()

In [65]:
query = '자연어 처리는 어렵습니다'
tokens = kiwi.tokenize(query)
tokens

[Token(form='자연어 처리', tag='NNP', start=0, len=6),
 Token(form='는', tag='JX', start=6, len=1),
 Token(form='어렵', tag='VA-I', start=8, len=2),
 Token(form='습니다', tag='EF', start=10, len=3)]

In [66]:
query = '이순신 장군'
tokens = kiwi.tokenize(query)
tokens

[Token(form='이순신', tag='NNP', start=0, len=3),
 Token(form='장군', tag='NNG', start=4, len=2)]

In [68]:
for t in tokens:
    if t.tag in ['NNP', 'NNG']:
        print(t.form)

이순신
장군


In [ ]:
# llm

In [69]:
documents = [
    "Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.",
    "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.",
    "GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.",
    "RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.",
    "FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워크입니다.",
    "트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.",
    "FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러리입니다.",
    "프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.",
    "임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.",
    "LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다.",
    "청킹(Chunking)은 긴 문서를 검색에 적합한 크기로 분할하는 기법입니다.",
]

In [70]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [71]:
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

In [74]:
docs = [Document(page_content=text, metadata={'doc_id' : i}) for i, text in enumerate(documents)]
vectorstore = FAISS.from_documents(docs, embeddings)

In [77]:
def _score(distance):
    return 1.0 / (1.0 + float(distance))

def search_faiss(query, top_k=3):
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc.metadata['doc_id'], _score(dist)) for doc, dist in results]

def search_by_vector(embedding, top_k=3):
    results = vectorstore.similarity_search_with_score_by_vector(embedding, k=top_k)
    return [(doc.metadata['doc_id'], _score(dist)) for doc, dist in results]

In [79]:
results = search_faiss('Python 프로그래밍')
results
for idx, score in results:
    print(f" [{idx}] {score} | {documents[idx][:30]}")

 [0] 0.49118694263501056 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
 [5] 0.4318095151812151 | FastAPI는 Python으로 빠른 웹 API를 구축
 [3] 0.4050654426863801 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 


In [ ]:
# 쿼리 확장
# ML 모델 학습 뭔가요? -> ML, 모델, 학습 / ML 모델 
# ML 모델 학습 뭔가요? ->  머신러닝 모델 뭔가요, ML 모델 종류, 머신러닝 개념

In [ ]:
# Multi-query : 여러 관점으로 생성
# HyDE  : Hypothesis DE : 질문 -> 가상답변을 생성 -> 검색
#                                 머신러닝은 기계로 ~~ 겁니다.
# Query-Decomposition : 쿼리를 나눈다 : 복잡한 질문 -> 여러개의 sub-query

In [80]:
query = "AI 챗봇 답변 품질 높이기"

In [82]:
results = search_faiss(query, top_k=3)
for idx, score in results:
    print(f" [{idx}] {score} | {documents[idx][:30]}")

 [7] 0.3945076736010113 | FAISS는 Facebook AI가 개발한 효율적인 유
 [4] 0.37909970706863505 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
 [2] 0.37676605269509683 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 


In [84]:
expaned_queries = [
    "AI 챗봇 답변 품질 높이기",
    "RAG 검색 증강 생성 기법",
    "프롬프트 엔지니어링 LLM입력 설계"
]

all_results = {}
for q in expaned_queries:
    results = search_faiss(q, top_k=3)
    print(f"{q} -> Top-1 : [{results[0][0]}] {documents[results[0][0]][:30]}")
    for idx, score in results:
        if idx not in all_results or score > all_results[idx]:
            all_results[idx] = score

AI 챗봇 답변 품질 높이기 -> Top-1 : [7] FAISS는 Facebook AI가 개발한 효율적인 유
RAG 검색 증강 생성 기법 -> Top-1 : [4] RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
프롬프트 엔지니어링 LLM입력 설계 -> Top-1 : [8] 프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는


In [85]:
ranked = sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:3]
for idx, score in ranked:
    print(f" [{idx}] {score} | {documents[idx][:30]}") 

 [8] 0.7814240425623532 | 프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는
 [4] 0.6104010978855806 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
 [11] 0.42203290813128236 | 청킹(Chunking)은 긴 문서를 검색에 적합한 크기


In [86]:
def search_with_expansion(original, expanded_list, top_k=5):
    all_results = {}
    for q in [original] + expanded_list:
        for idx, score in search_faiss(q, top_k=top_k):
            if idx not in all_results or score > all_results[idx]:
                all_results[idx] = score
                
    return sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:top_k]

In [87]:
original = "AI 챗봇 답변 품질 높이기"
expanded = ["RAG 검색 증강 생성 기법", "프롬프트 엔지니어링 LLM입력 설계"]

without = search_faiss(original, top_k=5)
with_exp = search_with_expansion(original, expanded, top_k=5)

without_ids = set(idx for idx, _ in without)
with_exp_ids = set(idx for idx, _ in with_exp)

new_doc = with_exp_ids - without_ids

print(f"without exp : {sorted(without_ids)}")
print(f"with exp : {sorted(with_exp_ids)}")
print(f"new : {sorted(new_doc)}")

without exp : [2, 4, 7, 9, 11]
with exp : [4, 6, 7, 8, 11]
new : [6, 8]


In [92]:
def search_with_rrf(queries, top_k=3, k=60):
    rrf_scores = {}
    for query in queries:
        results = search_faiss(query, top_k=5)
        for rank, (doc_id, _ ) in enumerate(results, 1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

In [93]:
queries = [original] + expanded
for idx, score in search_with_rrf(queries):
    print(f"[{idx}] {score} | {documents[idx][:30]}")

[4] 0.048651507139079855 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
[7] 0.032266458495966696 | FAISS는 Facebook AI가 개발한 효율적인 유
[8] 0.03177805800756621 | 프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는


In [ ]:
# 동의어 
# 구체화
# 관점

In [ ]:
# llm : 서로 다른 관점에서 ~~, 생성할 쿼리 수 (3~5), 원래 질문의 핵심의도도 유지

In [94]:
llm = ChatOpenAI(model ='gpt-4o-mini')

In [95]:
MULTI_QUERY_PROMPT = """주어진 질문을 3가지 서로 다른 관점에서 재작성하세요.
각 쿼리는 원래 질문의 의도를 유지하되, 다른 단어와 표현을 사용하세요.

원래 질문 : {query}

재작성1:
재작성2:
재작성3:"""

def generate_multi_queries(query, n=3):
    response = llm.invoke(MULTI_QUERY_PROMPT.format(query=query)).content
    queries = [line.strip() for line in response.strip().split('\n') if line.strip()]
    return queries[:n]

In [96]:
query = '벡터검색'
expanded = generate_multi_queries(query, n=3)
print(expanded)

['재작성1: 벡터 기반 검색', '재작성2: 벡터를 활용한 검색 방법', '재작성3: 벡터를 통한 정보 검색']


In [97]:
def multi_query_search(query, top_k=3):
    expanded = generate_multi_queries(query, n=3)
    all_queries = [query] + expanded
    
    return search_with_rrf(all_queries, top_k=top_k)

In [98]:
results = multi_query_search(query, top_k=3)
for idx, score in results:
    print(f"[{idx}] {score} | {documents[idx][:30]}")

[2] 0.06557377049180328 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 
[9] 0.06401209677419355 | 임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을
[11] 0.06349206349206349 | 청킹(Chunking)은 긴 문서를 검색에 적합한 크기


In [99]:
results = search_faiss(query, top_k=3)
for idx, score in results:
    print(f"[{idx}] {score} | {documents[idx][:30]}")

[2] 0.5392746966036289 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 
[4] 0.43729193908549846 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
[11] 0.4158700532996907 | 청킹(Chunking)은 긴 문서를 검색에 적합한 크기


In [103]:
def query_diversity(queries):
    if len(queries) <2:
        return 0
    
    embs = np.array(embeddings.embed_documents(queries))  # (4, 1536)
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    embs_noms = embs / norms
    
    total_sim, count = 0, 0
    for i in range(len(queries)):
        for j in range(i+1, len(queries)):
            total_sim += np.dot(embs_noms[i], embs_noms[j])
            count+=1
    
    avg_sim = total_sim / count
    diversity = 1-avg_sim
    
    return diversity

In [104]:
diverse = ["Python 웹 개발", "딥러닝 모델 구조", "데이터베이스 설계"]
similar = ["Python 웹 개발", "Python 웹 프레임워크", "Python 웹 서버"]

In [105]:
query_diversity(diverse)

np.float64(0.7951821094917046)

In [106]:
query_diversity(similar)

np.float64(0.179924355812482)

In [108]:
HYDE_PROMPT = """아래 질문에 대해 3~4문장으로 답변을 작성해주세요.
정확하지 않아도 괜찮습니다. 관련 주제의 문서처럼 작성하세요.

질문 {query}

답변:"""

In [111]:
def generate_hypothesis(query):
    return llm.invoke(HYDE_PROMPT.format(query=query)).content

In [112]:
query = 'RAG란 무엇인가요?'
generate_hypothesis(query)

'RAG는 "Retrieve and Generate"의 약자로, 인공지능 모델이 정보 검색과 생성 과정을 함께 활용하는 방법론을 의미합니다. 이 방법은 대량의 데이터에서 필요한 정보를 검색한 후, 해당 정보를 기반으로 새로운 내용을 생성하는 방식으로 작동합니다. RAG는 특히 대화형 AI 시스템, 챗봇 및 질의응답 시스템에서 유용하며, 사용자에게 보다 정확하고 배경 지식이 풍부한 답변을 제공하는 데 기여합니다. 이러한 접근 방식은 정보의 신뢰성과 생성된 콘텐츠의 품질을 동시에 향상시키는 데 중요한 역할을 합니다.'

In [113]:
def hyde_search(query, top_k=3):
    hypothesis = generate_hypothesis(query)
    
    hyp_emb = embeddings.embed_query(hypothesis)
    return search_by_vector(hyp_emb, top_k)

In [114]:
results = hyde_search('RAG란 무엇인가요?')

In [115]:
for idx, score in results:
    print(f"[{idx}] {score} | {documents[idx][:30]}")

[4] 0.5662832470682554 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
[1] 0.4697382379081969 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고
[3] 0.4348414866768689 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 


In [116]:
results = search_faiss('RAG란 무엇인가요?')
for idx, score in results:
    print(f"[{idx}] {score} | {documents[idx][:30]}")

[4] 0.5179503274217717 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
[10] 0.38717643479365377 | LangChain은 LLM 기반 애플리케이션 개발을 위
[7] 0.3807749979868685 | FAISS는 Facebook AI가 개발한 효율적인 유
